# PHẦN V — BOOTSTRAP CEPH CLUSTER (RUNBOOK RÚT GỌN)

Notebook rút gọn: chỉ lệnh + comment ngắn.

## BƯỚC 19 — Cài tool Ceph trên 3 server

In [ ]:
sudo apt update
sudo apt install -y ceph-common lvm2 gdisk smartmontools chrony curl wget openssh-client openssh-server
sudo systemctl enable --now chrony
chronyc tracking

## BƯỚC 19 — /etc/hosts trên 3 server

In [ ]:
sudo cp /etc/hosts /etc/hosts.bak.$(date +%F-%H%M%S)

sudo tee -a /etc/hosts << EOF
10.0.0.21  server-a
10.0.0.22  server-b
10.0.0.23  server-c
EOF

getent hosts server-a
getent hosts server-b
getent hosts server-c

## BƯỚC 19 — SSH từ Server A sang B/C

In [ ]:
# Chạy trên Server A
ssh-keygen -t ed25519 -f ~/.ssh/ceph_deploy -C "ceph-deploy@lab" -N ""
ssh-copy-id -i ~/.ssh/ceph_deploy.pub ubuntu@10.0.0.22
ssh-copy-id -i ~/.ssh/ceph_deploy.pub ubuntu@10.0.0.23

ssh -i ~/.ssh/ceph_deploy ubuntu@10.0.0.22 hostname
ssh -i ~/.ssh/ceph_deploy ubuntu@10.0.0.23 hostname

## BƯỚC 20 — Check OSD disk /dev/sda5

In [ ]:
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINTS /dev/sda
sudo pvs
sudo vgs
sudo lvs

# /dev/sda5 là OSD candidate, không wipe sda3/sda4
lsblk /dev/sda5
sudo wipefs -n /dev/sda5
findmnt /dev/sda5 || true
sudo pvs | grep sda5 || true

## BƯỚC 20 — Wipe OSD /dev/sda5

In [ ]:
# Chạy trên cả 3 server, chỉ khi chắc /dev/sda5 là OSD
sudo wipefs -a /dev/sda5
sudo sgdisk --zap-all /dev/sda5 || true
sudo partprobe /dev/sda || true

sudo wipefs -n /dev/sda5
lsblk -f /dev/sda5

## BƯỚC 21 — Cài cephadm Server A

In [ ]:
# Chỉ chạy trên Server A
curl --silent --remote-name --location https://github.com/ceph/ceph/raw/reef/src/cephadm/cephadm
chmod +x cephadm
sudo ./cephadm add-repo --release reef
sudo ./cephadm install

which cephadm
cephadm version || sudo cephadm version

## BƯỚC 21 — Bootstrap Ceph mgmt network

In [ ]:
# Chọn 1: bootstrap bằng 10.0.0.x
sudo cephadm bootstrap \
  --mon-ip 10.0.0.21 \
  --cluster-network 10.0.0.0/24 \
  --initial-dashboard-user admin \
  --initial-dashboard-password 'Admin@Ceph2025!'

sudo ceph -s
sudo ceph mon stat
sudo ceph mgr services

## BƯỚC 21 — Bootstrap Ceph cluster network

In [ ]:
# Chọn 2: bootstrap bằng 10.0.3.x nếu br-cluster ổn
ping -c 3 10.0.3.22
ping -c 3 10.0.3.23

sudo cephadm bootstrap \
  --mon-ip 10.0.3.21 \
  --cluster-network 10.0.3.0/24 \
  --initial-dashboard-user admin \
  --initial-dashboard-password 'Admin@Ceph2025!'

sudo ceph -s
sudo ceph mon stat
sudo ceph mgr services

## BƯỚC 22 — Add Server B/C

In [ ]:
# Dùng IP đúng với network bootstrap
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.0.22
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.0.23

sudo ceph orch host add server-b 10.0.0.22
sudo ceph orch host add server-c 10.0.0.23

sudo ceph orch host ls

## BƯỚC 22 — Deploy MON/MGR

In [ ]:
sudo ceph orch apply mon 3
sudo ceph orch ps --daemon-type mon
sudo ceph mon stat

sudo ceph orch apply mgr 2
sudo ceph orch ps --daemon-type mgr
sudo ceph mgr stat

## BƯỚC 23 — Add OSD

In [ ]:
sudo ceph orch device ls --wide

sudo ceph orch daemon add osd server-a:/dev/sda5
sudo ceph orch daemon add osd server-b:/dev/sda5
sudo ceph orch daemon add osd server-c:/dev/sda5

sudo ceph orch ps --daemon-type osd
sudo ceph osd tree
sudo ceph -s

## BƯỚC 23 — Tạo OpenStack pools

In [ ]:
sudo ceph osd pool create images 32
sudo ceph osd pool create volumes 32
sudo ceph osd pool create vms 32
sudo ceph osd pool create backups 16

sudo ceph osd pool application enable images rbd
sudo ceph osd pool application enable volumes rbd
sudo ceph osd pool application enable vms rbd
sudo ceph osd pool application enable backups rbd

sudo ceph osd pool set images size 3
sudo ceph osd pool set volumes size 3
sudo ceph osd pool set vms size 3
sudo ceph osd pool set backups size 3

sudo ceph osd pool set images min_size 2
sudo ceph osd pool set volumes min_size 2
sudo ceph osd pool set vms min_size 2
sudo ceph osd pool set backups min_size 2

sudo ceph osd pool ls
sudo ceph df

## BƯỚC 23 — Tạo OpenStack Ceph users

In [ ]:
sudo ceph auth get-or-create client.glance \
  mon "profile rbd" \
  osd "profile rbd pool=images" \
  > /etc/ceph/ceph.client.glance.keyring

sudo ceph auth get-or-create client.cinder \
  mon "profile rbd" \
  osd "profile rbd pool=volumes, profile rbd pool=vms, profile rbd pool=images" \
  > /etc/ceph/ceph.client.cinder.keyring

sudo ceph auth get-or-create client.nova \
  mon "profile rbd" \
  osd "profile rbd pool=vms, profile rbd pool=images" \
  > /etc/ceph/ceph.client.nova.keyring

sudo chmod 600 /etc/ceph/ceph.client.*.keyring
sudo ceph auth get client.glance
sudo ceph auth get client.cinder
sudo ceph auth get client.nova

## BƯỚC 23 — Copy Ceph config sang Controllers

In [ ]:
sudo scp /etc/ceph/ceph.conf ubuntu@10.0.0.11:/tmp/
sudo scp /etc/ceph/ceph.client.*.keyring ubuntu@10.0.0.11:/tmp/

sudo scp /etc/ceph/ceph.conf ubuntu@10.0.0.12:/tmp/
sudo scp /etc/ceph/ceph.client.*.keyring ubuntu@10.0.0.12:/tmp/

sudo scp /etc/ceph/ceph.conf ubuntu@10.0.0.13:/tmp/
sudo scp /etc/ceph/ceph.client.*.keyring ubuntu@10.0.0.13:/tmp/

## BƯỚC 24 — Verify Ceph

In [ ]:
sudo ceph -s
sudo ceph health detail
sudo ceph orch host ls
sudo ceph mon stat
sudo ceph mgr stat
sudo ceph osd tree
sudo ceph osd df
sudo ceph osd pool ls
sudo ceph mgr services